<a href="https://colab.research.google.com/github/veetmoradiya3628/Data-engineering-learning/blob/main/pyspark/PySpark_Data_Manipulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark

In [4]:
df = spark.read.parquet('/content/yellow_tripdata_2025-01.parquet')

In [5]:
from pyspark.sql.functions import col, isnull


df.filter(isnull(col('fare_amount'))).count()

0

In [6]:
df.filter(isnull(col('passenger_count'))).count()

540149

In [7]:
df1 = df.fillna({'passenger_count': 1})
df1.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

In [8]:
df1.filter(isnull(col('passenger_count'))).count()

0

In [9]:
from pyspark.sql.functions import unix_timestamp, round

df1 = df.withColumn(
    'trip_duration_minutes',
    round((unix_timestamp(col('tpep_dropoff_datetime')) - unix_timestamp(col('tpep_pickup_datetime')))/60, 1)
)

In [12]:
df1.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|trip_duration_minutes|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|           

In [11]:
df1.drop('VendorID', 'RatecodeID').show(5)

+--------------------+---------------------+---------------+-------------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|trip_duration_minutes|
+--------------------+---------------------+---------------+-------------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------------+
| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|                 N|         229|         237|     

In [14]:
df_feb = spark.read.parquet('/content/yellow_tripdata_2025-02.parquet')
df_jan = spark.read.parquet('/content/yellow_tripdata_2025-01.parquet')

df_2025 = df_jan.union(df_feb)
df_2025.count()

7052769

In [15]:
taxi_zone_lookup = spark.read.option('header', 'true').csv('/content/taxi_zone_lookup.csv')
taxi_zone_lookup.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [16]:
df_joined = df_2025.join(taxi_zone_lookup, df_2025.PULocationID == taxi_zone_lookup.LocationID, 'left')
df_joined.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+--------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|LocationID|  Borough|                Zone|service_zone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+

In [17]:
df.groupBy('payment_type').count().sort('payment_type').show()

+------------+-------+
|payment_type|  count|
+------------+-------+
|           0| 540149|
|           1|2444393|
|           2| 390429|
|           3|  23773|
|           4|  76481|
|           5|      1|
+------------+-------+



In [18]:
df.groupBy('payment_type').avg('total_amount').show()

+------------+------------------+
|payment_type| avg(total_amount)|
+------------+------------------+
|           5|               0.0|
|           1|28.079206040112336|
|           3| 6.145837294409612|
|           2| 21.74035097290552|
|           4|11.573446869157074|
|           0|20.085374294870405|
+------------+------------------+



In [20]:
from pyspark.sql.functions import avg

df.groupBy('payment_type').agg(avg('total_amount').alias('avg_amount')).show()


+------------+------------------+
|payment_type|        avg_amount|
+------------+------------------+
|           5|               0.0|
|           1|28.079206040112336|
|           3| 6.145837294409612|
|           2| 21.74035097290552|
|           4|11.573446869157074|
|           0|20.085374294870405|
+------------+------------------+



In [21]:
avg_fare = df.groupBy('payment_type').agg(avg('total_amount')).sort('payment_type')
avg_fare.write.csv('/content/avg_fare', header=True, mode='overwrite')

In [24]:
# PySpark SQL

taxi = spark.read.parquet('/content/yellow_tripdata_2025-01.parquet')

In [25]:
taxi.createOrReplaceTempView('taxi')

In [26]:
spark.sql('SELECT * FROM taxi WHERE total_amount > 50').show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-01-01 00:15:41|  2025-01-01 01:03:03|              4|         3.05|         1|                 N|         114|    

In [29]:
spark.sql("""
SELECT
  payment_type,
  passenger_count,
  total_amount
FROM
  taxi
WHERE
  total_amount > 50 AND passenger_count > 2
""") \
.show(5)

+------------+---------------+------------+
|payment_type|passenger_count|total_amount|
+------------+---------------+------------+
|           1|              4|       50.76|
|           2|              9|      111.32|
|           1|              3|       56.38|
|           1|              4|        58.3|
|           1|              3|       51.55|
+------------+---------------+------------+
only showing top 5 rows


In [28]:
query = '''
SELECT
  payment_type,
  passenger_count,
  total_amount
FROM
  taxi
WHERE
  total_amount > 50
  AND
  passenger_count > 2
LIMIT
  5
'''
spark.sql(query).show()

+------------+---------------+------------+
|payment_type|passenger_count|total_amount|
+------------+---------------+------------+
|           1|              4|       50.76|
|           2|              9|      111.32|
|           1|              3|       56.38|
|           1|              4|        58.3|
|           1|              3|       51.55|
+------------+---------------+------------+



#### Production environment requirement
- Should be scalable, reliable & secure

1. Data sources
- operational databases, application logs, APIs, flat files from vendors
- data extraction tools : fivetran, airbyte, kafka kinesis, custom scripts..

2. Distributed Storage
- HDFS
- Amazon S3
- Google Cloud Storage

3. Cluster Management
- YARN
- Kubernetes

4. Job Scheduling
- Apache Airflow

5. Consistent environments
- Docker or virtual environments

6. Monitoring and Logging
- Spark Web UI
- Grafana Dashboards
- AWS CloudWatch

7. Security and Access Control
- Setup IAM roles
- Network protection
- Encryption

#### Cloud services
- AWS EMR
- Dataproc
- Databricks
- Azure Synapse
- AWS Glue